In [1]:
#PREVENDO RENOVAÇÃO DE ASSINATURA DE STREAMING
#IMPORTANTO BIBLIOTECAS NECESSÁRIAS
import pandas as pd

import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

#1 - CRIANDO DADOS FICTÍCIOS
#Criando clientes com: meses de uso, horas de uso/mes e número de reclamações
np.random.seed(42)

quantidade_clientes = 300

meses_cliente = np.random.randint(1,61, quantidade_clientes)
horas_uso_mes = np.random.randint(1,51, quantidade_clientes)
reclamacoes = np.random.randint(0,8, quantidade_clientes)

#2 - CRIANDO O TARGET
#cria uma lista com valores booleanos (se atender condições, 1. Caso contrário, 0)
renovaria_assinatura = (
    (horas_uso_mes > 15) & (reclamacoes < 3)
).astype(int)

#3 - ADICIONANDO RUIDO ALEATÓRIO
#essa linha escolhe 10% dos clientes
ruido = np.random.rand(quantidade_clientes) < 0.10

#pega esses 10% e inverte. Se ele fosse cancelar > mantém. Se iria manter > cancela
renovaria_assinatura[ruido] = 1 - renovaria_assinatura[ruido]

#4 - CRIANDO O DATAFRAME
dados = pd.DataFrame({
    "meses_cliente": meses_cliente,
    "horas_uso_mes": horas_uso_mes,
    "reclamacoes": reclamacoes,
    "renovaria_assinatura": renovaria_assinatura
})

#5 - SEPARANDO FEATURES E TARGET
X = dados[
    [
        "meses_cliente",
        "horas_uso_mes",
        "reclamacoes"
    ]
]
y = dados["renovaria_assinatura"]

#6 - SEPARANDO TREINO E TESTE
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#7 - CRIANDO O MODELO
modelo = XGBClassifier(
    n_estimators = 100, #n de árvores
    max_depth=3, #n de decisões
    learning_rate = 0.1,
    random_state=42,
    eval_metric="logloss"
)

#8 - TREINANDO O MODELO
modelo.fit(X_treino, y_treino)

#9 - FAZENDO PREVISÕES
previsoes = modelo.predict(X_teste)

#10 - CONFERINDO TAXA DE SUCESSO
acuracia = accuracy_score(y_teste, previsoes)

print(f'Acurácia do modelo: {acuracia:.2%}')

Acurácia do modelo: 88.33%
